# Maximal Marginal Relevance (MMR): Enhancing Retrieval Diversity in Advanced RAG Systems

Retrieval-Augmented Generation (RAG) systems are foundational to modern LLM applications, providing grounding and reducing hallucinations. While standard vector similarity search is highly effective at finding documents *related* to a query, it suffers from a critical flaw: redundancy. When source documents cover the same concept using slightly different phrasing—such as multiple descriptions of "gradient descent"—a simple similarity retriever may return several near-identical results, overwhelming the LLM with repetitive context and diluting the signal.

Maximal Marginal Relevance (MMR) is an advanced retrieval technique designed to solve this exact problem. MMR balances two objectives: maximizing the relevance of retrieved documents to the original query, while simultaneously ensuring that the selected set of documents are maximally diverse from one another. Instead of simply picking the $K$ most similar chunks, MMR iteratively selects a document that is highly relevant *and* least similar to the documents already chosen.

For advanced RAG pipelines and complex LangGraph agents, diversity is as important as relevance. An agent might need not just "information about deep learning," but rather a comprehensive view covering different facets: one chunk on optimization (gradient descent), one on regularization (Dropout), and one on stability (Batch Norm). By mastering MMR, developers can build robust retrieval components that provide rich, non-overlapping context, leading to more accurate, nuanced, and actionable LLM outputs.

### Learning Objectives

Upon completing this notebook, you will be able to:

*   **Identify Retrieval Limitations:** Understand the concept of document redundancy in standard vector similarity search results.
*   **Grasp MMR Theory:** Explain how Maximal Marginal Relevance (MMR) balances query relevance with source document diversity.
*   **Implement Advanced Retrievers:** Utilize specialized retrieval methods (like MMR) to select a set of context documents that are both highly relevant and maximally diverse.
*   **Optimize RAG Context:** Apply the principles of diversity-aware retrieval to build more robust and informative grounding for advanced LLM applications using LangGraph.


### Setup and Environment Loading

This cell initializes the environment by loading API keys (like `OPENAI_API_KEY`) from a `.env` file using `dotenv`. It also imports necessary components for embedding generation (`OpenAIEmbeddings`), vector storage (`Chroma`), and document handling (`Document`).


In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Load OPENAI_API_KEY from .env file into the environment variables
load_dotenv()


True

In [3]:
# 15 documents: the first 3 deep learning docs are intentionally near-identical —
# all describe gradient descent as the core optimisation technique, with minor phrasing variation.
# The next 3 deep learning docs cover distinct regularisation/training concepts.
# At lambda_mult=1.0 (pure relevance), a query about deep learning training returns
# all 3 near-identical gradient descent docs, showing the redundancy problem.
# As lambda_mult decreases, MMR penalises already-selected similar docs and
# picks one gradient descent doc + Dropout + Batch Norm + LR Scheduler instead.
docs = [
    Document(page_content="Training a deep learning model involves iteratively adjusting weights using gradient descent to minimise the loss.", metadata={"topic": "deep learning"}),
    Document(page_content="Deep learning models are optimised through gradient descent, which updates weights in the direction that reduces the training loss.", metadata={"topic": "deep learning"}),
    Document(page_content="Gradient descent is the core optimisation technique in deep learning, guiding weight updates based on computed gradients of the loss.", metadata={"topic": "deep learning"}),
    Document(page_content="Dropout randomly disables a fraction of neurons during training to prevent overfitting in deep networks.", metadata={"topic": "deep learning"}),
    Document(page_content="Batch normalisation stabilises training by normalising layer inputs, which allows the use of higher learning rates.", metadata={"topic": "deep learning"}),
    Document(page_content="Learning rate schedulers dynamically adjust the learning rate during training to improve convergence and avoid overshooting.", metadata={"topic": "deep learning"}),
    Document(page_content="Arctic sea ice has declined by about 13% per decade since satellite measurements began in 1979.", metadata={"topic": "climate"}),
    Document(page_content="Carbon capture technology removes CO2 from the atmosphere and stores it underground.", metadata={"topic": "climate"}),
    Document(page_content="The permafrost in Siberia contains vast amounts of methane that could be released as it thaws.", metadata={"topic": "climate"}),
    Document(page_content="The Renaissance was a cultural movement in Europe from the 14th to 17th century that revived classical art.", metadata={"topic": "art"}),
    Document(page_content="Impressionism emerged in 19th-century France, focusing on light, colour, and everyday subjects.", metadata={"topic": "art"}),
    Document(page_content="Abstract expressionism prioritises spontaneous, automatic, and subconscious creation.", metadata={"topic": "art"}),
    Document(page_content="Common law systems derive legal principles from judicial precedent rather than written codes.", metadata={"topic": "law"}),
    Document(page_content="The presumption of innocence requires the prosecution to prove guilt beyond reasonable doubt.", metadata={"topic": "law"}),
    Document(page_content="Intellectual property law protects creations of the mind, including patents, trademarks, and copyrights.", metadata={"topic": "law"}),
]

### Vector Store Initialization (Embedding and Storage)

This cell initializes the embedding process using `OpenAIEmbeddings` and then populates a ChromaDB vector store. It converts the list of loaded documents (`docs`) into numerical embeddings and stores them in a persistent database collection named `mmr_demo`, making them retrievable for subsequent RAG steps.


In [4]:
# Embed documents and store in ChromaDB
# Initialize the embedding model using OpenAI's text-embedding-3-small.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Create a Chroma vectorstore by passing the list of document objects (docs).
# The 'embedding' argument specifies which function to use for generating vectors.
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="mmr_demo",
)


### Code Explanation

This cell demonstrates the core retrieval step of RAG. It initializes a `sim_retriever` using the existing vector store, configuring it to retrieve the top 3 most similar documents based on the input query. Finally, it executes the retriever and prints the topic and content of each retrieved document.


In [7]:
query = "deep learning model training and its optimization techniques"

sim_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

results = sim_retriever.invoke(query)

topics = [doc.metadata["topic"] for doc in results]
    
for i, doc in enumerate(results, 1):
    print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")
    print()


  [1] topic=deep learning: Deep learning models are optimised through gradient descent, which updates weights in the direction that reduces the training loss.

  [2] topic=deep learning: Training a deep learning model involves iteratively adjusting weights using gradient descent to minimise the loss.

  [3] topic=deep learning: Gradient descent is the core optimisation technique in deep learning, guiding weight updates based on computed gradients of the loss.



### MMR Hyperparameter Tuning (Relevance-Diversity Trade-off)

This cell demonstrates the effect of the `lambda_mult` parameter in Maximum Marginal Relevance (MMR) search. By iterating through different lambda values, we systematically tune the balance between retrieving highly relevant documents (high $\lambda$) and ensuring the retrieved set is diverse and covers various topics (low $\lambda$).


In [8]:
query = "deep learning model training and its optimization techniques"

# lambda_mult controls the relevance-diversity trade-off:
#   1.0 = pure relevance (identical to similarity search)
#   0.0 = pure diversity (ignores relevance entirely)
# fetch_k: number of candidate docs fetched before MMR re-ranks and selects k
lambda_values = [1.0, 0.7, 0.5 ,0.0]

for lm in lambda_values: # Iterate through different lambda values to test the trade-off
    # Initialize the retriever using the 'mmr' search type
    retriever = vectorstore.as_retriever(
        search_type="mmr",
        # Define search parameters: k=3 (select 3 docs), fetch_k=10 (check top 10 candidates)
        search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": lm},
    )
    # Invoke the retriever with the query
    results = retriever.invoke(query)
    
    # Extract topics from the retrieved documents for display
topics = [doc.metadata["topic"] for doc in results]
    print(f"=== lambda_mult={lm} ===") # Print the current lambda value being tested
    for i, doc in enumerate(results, 1): # Loop through and print each retrieved document
        print(f"  [{i}] topic={doc.metadata['topic']}: {doc.page_content}")
    print()



=== lambda_mult=1.0 ===
  [1] topic=deep learning: Deep learning models are optimised through gradient descent, which updates weights in the direction that reduces the training loss.
  [2] topic=deep learning: Training a deep learning model involves iteratively adjusting weights using gradient descent to minimise the loss.
  [3] topic=deep learning: Gradient descent is the core optimisation technique in deep learning, guiding weight updates based on computed gradients of the loss.

=== lambda_mult=0.7 ===
  [1] topic=deep learning: Deep learning models are optimised through gradient descent, which updates weights in the direction that reduces the training loss.
  [2] topic=deep learning: Learning rate schedulers dynamically adjust the learning rate during training to improve convergence and avoid overshooting.
  [3] topic=deep learning: Dropout randomly disables a fraction of neurons during training to prevent overfitting in deep networks.

=== lambda_mult=0.5 ===
  [1] topic=deep lear